In [2]:
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef
)

In [3]:
train_df = pd.read_csv("../data/processed_data/train.csv")
dev_df = pd.read_csv("../data/processed_data/dev.csv")
train_df.head()

,Unnamed: 0.1,Unnamed: 0,PID,Text_data,Label
0,2782,9097,dev_pid_207,Finally Leaving : I finally left my husband. W...,moderate
1,3794,10110,dev_pid_1220,Hate... : I hate myself! I hate my stupid fat ...,moderate
2,5080,11495,dev_pid_2605,"I'm just a burden. : I'm annoying, stupid, ugl...",not depression
3,6210,12711,dev_pid_3821,Self-Image : I wish sometimes that I wasn’t se...,not depression
4,3872,10188,dev_pid_1298,I find it kind of funny seeing people freak ou...,moderate


In [4]:
label2id = {
    "not depression": 0,
    "moderate": 1,
    "severe": 2
}

id2label = {v: k for k, v in label2id.items()}

train_df["label"] = train_df["Label"].map(label2id)
dev_df["label"] = dev_df["Label"].map(label2id)

print(train_df["label"].value_counts())
print("\nUnmapped training labels:", train_df["label"].isna().sum())
print("Unmapped dev labels:", dev_df["label"].isna().sum())

label
1    2807
0    2116
2     565
Name: count, dtype: int64

Unmapped training labels: 0
Unmapped dev labels: 0


In [5]:
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
MAX_LENGTH = 256

train_encodings = tokenizer(
    train_df["Text_data"].astype(str).tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

dev_encodings = tokenizer(
    dev_df["Text_data"].astype(str).tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

In [7]:
class DepressionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(int(self.labels[idx]))
        return item


train_dataset = DepressionDataset(
    train_encodings, train_df["label"].tolist()
)

dev_dataset = DepressionDataset(
    dev_encodings, dev_df["label"].tolist()
)

print("Training samples:", len(train_dataset))
print("Development samples:", len(dev_dataset))

Training samples: 5488
Development samples: 1373


In [8]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Using device:", device)

Using device: mps


In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

model.to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "mcc": matthews_corrcoef(labels, predictions)
    }

In [12]:
training_args = TrainingArguments(
    output_dir="./distilbert_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=14,
    save_total_limit=2
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

In [14]:
train_results = trainer.train()

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Balanced Accuracy,Mcc
1,3.028797,0.802649,0.611071,0.551881,0.591933,0.548075,0.315963
2,2.783391,0.789271,0.621267,0.581280,0.617229,0.580750,0.339126
3,2.213936,0.828413,0.633649,0.594298,0.631572,0.585600,0.358081


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/processed_data/train.csv")

train_labels = np.array([
    label2id[label] for label in train_df["Label"]
])

print("Training samples:", len(train_labels))
print("Class distribution:", np.bincount(train_labels))
print("Label mapping:", label2id)

Training samples: 5488
Class distribution: [2116 2807  565]
Label mapping: {'not depression': 0, 'moderate': 1, 'severe': 2}


In [17]:
import torch
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(train_labels)
)

class_weights = torch.tensor(class_weights, dtype=torch.float32)

print("Class mapping:", label2id)
print("Class counts:", np.bincount(np.array(train_labels).astype(int)))
print("Class weights:", class_weights)

Class mapping: {'not depression': 0, 'moderate': 1, 'severe': 2}
Class counts: [2116 2807  565]
Class weights: tensor([0.8645, 0.6517, 3.2378])


In [18]:
from pathlib import Path

weighted_output_dir = "./distilbert_weighted_results"
Path(weighted_output_dir).mkdir(parents=True, exist_ok=True)

print("Weighted model output directory:", weighted_output_dir)

Weighted model output directory: ./distilbert_weighted_results


In [19]:
training_args = TrainingArguments(
    output_dir="./distilbert_weighted_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=14,
    save_total_limit=2
)

In [20]:
import inspect
from transformers import Trainer

print(inspect.signature(Trainer.__init__))

(self, model: transformers.modeling_utils.PreTrainedModel | torch.nn.modules.module.Module | None = None, args: transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: 'Dataset | IterableDataset | datasets.Dataset | None' = None, eval_dataset: 'Dataset | dict[str, Dataset] | datasets.Dataset | None' = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.image_processing_utils.BaseImageProcessor | transformers.feature_extraction_utils.FeatureExtractionMixin | transformers.processing_utils.ProcessorMixin | None = None, model_init: collections.abc.Callable[..., transformers.modeling_utils.PreTrainedModel] | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callbac

In [21]:
import torch.nn.functional as F

def weighted_loss_func(outputs, labels, num_items_in_batch=None):
    logits = outputs.logits

    weights = class_weights.to(logits.device)

    return F.cross_entropy(
        logits,
        labels,
        weight=weights
    )

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    compute_loss_func=weighted_loss_func
)

print("Weighted DistilBERT Trainer initialized.")
print("Output directory:", training_args.output_dir)
print("Class weights:", class_weights)

Weighted DistilBERT Trainer initialized.
Output directory: ./distilbert_weighted_results
Class weights: tensor([0.8645, 0.6517, 3.2378])


In [23]:
train_result = trainer.train()

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Balanced Accuracy,Mcc
1,1.855271,1.045486,0.607429,0.577792,0.608184,0.583961,0.326836
2,1.338905,1.541724,0.597961,0.568780,0.597999,0.566283,0.306637
3,1.211334,2.054007,0.604516,0.569411,0.603370,0.559854,0.309810


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]